# 0 · Setup — auto-detect everything + enter your HF token once

Role, region, and the S3 bucket (SageMaker's default bucket) are **auto-detected** — no placeholders. Your Hugging Face token is entered **once here** like a password (`getpass`) and reused by every job. No Secrets Manager.

Run the cells; this writes `toolcall_config.json` for notebooks 01–04.

> The token is stored in `toolcall_config.json` on this instance so later notebooks can use it. Don't commit that file (add it to `.gitignore`).

In [ ]:
# pip install -r ../requirements-launch.txt

In [ ]:
import json, time
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
region = sess.boto_region_name

try:
    role = get_execution_role()
except Exception as e:
    raise RuntimeError(
        'Could not auto-detect an execution role. Run this in a SageMaker '
        'notebook, or set role manually to your SageMaker role ARN. '
        f'Original error: {e}')

bucket = sess.default_bucket()            # auto: sagemaker-{region}-{account}
try:
    account = sess.account_id()
except Exception:
    import boto3
    account = boto3.client('sts', region_name=region).get_caller_identity()['Account']

run_id = time.strftime('%Y%m%d-%H%M%S')
S3_BASE = f's3://{bucket}/toolcall/{run_id}'

print('region :', region)
print('account:', account)
print('role   :', role)
print('bucket :', bucket, '(SageMaker default)')
print('run    :', S3_BASE)

### Enter your Hugging Face token (hidden input)
Needed for gated models/datasets (xLAM, Qwen). Get one at https://huggingface.co/settings/tokens.

In [ ]:
import getpass, os
HF_TOKEN = os.environ.get('HF_TOKEN', '').strip()
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Hugging Face token (input hidden): ').strip()
assert HF_TOKEN, 'A Hugging Face token is required.'

# Verify it works before we bake it into the config.
from huggingface_hub import login, whoami
login(token=HF_TOKEN, add_to_git_credential=False)
print('HF token OK for user:', whoami(token=HF_TOKEN)['name'])

### Choices you may want to change (working defaults; not placeholders)

In [ ]:
BASE_MODEL = 'Qwen/Qwen3.5-4B'          # or 'Qwen/Qwen3.5-9B'
GPU_INSTANCE = 'ml.p4de.24xlarge'       # 8x A100 80GB; p5.48xlarge also fine
CPU_INSTANCE = 'ml.m5.2xlarge'          # data-prep step only
GPU_IMAGE_URI = ''                      # '' -> resolve a stock PyTorch DLC

GPUS_PER_INSTANCE = {
    'ml.p4d.24xlarge': 8, 'ml.p4de.24xlarge': 8, 'ml.p5.48xlarge': 8,
    'ml.p5en.48xlarge': 8, 'ml.p3.8xlarge': 4, 'ml.p3.16xlarge': 8,
    'ml.g5.12xlarge': 4, 'ml.g5.24xlarge': 4, 'ml.g5.48xlarge': 8,
    'ml.g6e.12xlarge': 4, 'ml.g6e.24xlarge': 4, 'ml.g6e.48xlarge': 8,
}
NUM_GPUS = GPUS_PER_INSTANCE.get(GPU_INSTANCE)
if NUM_GPUS is None:
    NUM_GPUS = 8
    print(f'WARN: unknown GPU count for {GPU_INSTANCE}; defaulting to {NUM_GPUS}.')
print(f'{GPU_INSTANCE} -> NUM_GPUS={NUM_GPUS}')

### Write the config

In [ ]:
cfg = {
    'region': region, 'role': role, 'account': account,
    'hf_token': HF_TOKEN,
    'gpu_instance': GPU_INSTANCE, 'cpu_instance': CPU_INSTANCE,
    'num_gpus': NUM_GPUS, 'pt_version': '2.7', 'gpu_image_uri': GPU_IMAGE_URI,
    'base_model': BASE_MODEL,
    'xlam_dataset': 'Salesforce/xlam-function-calling-60k',
    'n_train': 8000, 'n_grpo': 3000, 'n_eval': 500,
    'seq_len': 2048, 'seed': 42, 'enable_thinking': 0, 'attn_impl': 'flash_attention_2',
    'use_lora': 1, 'lora_r': 16, 'lora_alpha': 32,
    'sft_epochs': 1, 'sft_lr': '2e-4', 'sft_bs': 2, 'sft_grad_accum': 8,
    'grpo_epochs': 1, 'grpo_lr': '1e-6', 'grpo_bs': 8, 'grpo_grad_accum': 4,
    'num_generations': 8, 'grpo_beta': 0.04, 'grpo_temperature': 1.0,
    'max_completion_length': 256, 'reward_weights': '0.1,0.3,0.2,0.4', 'use_vllm': 0,
    'eval_base': 1, 'eval_max_new_tokens': 256, 'eval_batch_size': 16,
    's3_base': S3_BASE,
    'data_s3': S3_BASE + '/data',
    'sft_model_s3': S3_BASE + '/sft_model',
    'grpo_model_s3': S3_BASE + '/grpo_model',
    'results_s3': S3_BASE + '/results',
}

_glob = cfg['grpo_bs'] * cfg['num_gpus'] * cfg['grpo_grad_accum']
assert _glob % cfg['num_generations'] == 0, (
    f"GRPO batch {_glob} not divisible by num_generations {cfg['num_generations']}.")

json.dump(cfg, open('toolcall_config.json', 'w'), indent=2)
redacted = {**cfg, 'hf_token': '***'}
print(json.dumps(redacted, indent=2))
print('\nwrote toolcall_config.json (token redacted above)')

Nothing to replace — role/region/bucket auto-detected, token entered once. **Next:** `01_prepare.ipynb`.